# Chapter 4: Implementing a GPT Model from Scratch

I'm building intuition for the full GPT Architecture by constructing the `GPTModel` class and learning its generative loop piecemeal. Here, I'll practice assembling the previously-learned Multi-Head Attention layer alongside new critical components like `LayerNorm` and `FeedForward`.

In [ ]:
import torch
import torch.nn as nn
import tiktoken

# Ensures reproducibility per the learning guidelines
torch.manual_seed(42)
print("Ready to practice PyTorch!")

## 1. Setting up the GPT Configurations

To start, I'll define a dictionary holding all the hyper-parameters for a GPT-2 Small sized model. This way I can change the overall shape centrally.

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "ctx_len": 1024,        # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

## 2. Implementing the Missing Components

I need a few missing Lego pieces before I assemble the full architecture: `LayerNorm`, `Activate`, and a `FeedForward` loop.

In [ ]:
class DummyLayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

# Test it out
test_tensor = torch.rand(2, 5, 768)
ln = DummyLayerNorm(768)
print(ln(test_tensor).shape)

## 3. The Generating Function

Finally, let's practice simple greedy decoding. My understanding is that we iteratively append the next highest probability token to the context sequence.